## LDA Topic Modelling on a Nigerian News Corpus

### Introduction & Overview

Topic modelling is a fascinating area of NLP that enables computers to classify and group text based on their topics. Topic Modelling is useful for parsing large text and relaying to users the dominant themes within it.
<br> In this case, we are deploying a Latent Dirichlet Allocation or LDA model. LDA is an algorithm that can parse text and output probabilistic word clusters. While the topics are not labelled semantically, LDA uses a numeric label to the word groupings. LDA operates on _two_ assumptions:
1. Documents (or in this case articles) are composed of topics
2. Topics are composed of words.

Training LDA aims to infer the underlying topics and their corresponding word distribution.

### Dataset Overview

I implemented an LDA model on a collection of news articles covering news events in Nigeria. The training set was sourced from a collection of Nigerian Newsfeed [articles](https://www.kaggle.com/datasets/prakharrathi25/nigeria-newsfeed/data). Thanks to [Prakhar Rathi](https://www.kaggle.com/prakharrathi25) for  putting this together!

The collection ranges across election news to crime reports and general current affairs. After training, the model should be able to group words used in similar contexts together and present a concise approximation of topics in the training corpus.

### Tools Used

* SpaCy
* Gensim
* PyLDAvis

### Model Build & Design

I started by loading the relevant libraries and dataset.

In [12]:
# Loading the libraries and dependencies
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import spacy
import gensim

from gensim import models, corpora

#### Load the Dataset

In [14]:
# Loading the Dataset
df_newsfeed = pd.read_csv("Nigeria2019_Newsfeed.csv")
newsfeed_article = df_newsfeed['Newsfeed_Description2'].to_list() # 'Newsfeed_Description2' column contains the article
print(len(newsfeed_article))

24493


The dataset contains 24,493 raw news records, providing sufficient volume for topic convergence.

#### Data Cleaning

To ensure that we are capturing useful information, I filtered out short articles. These articles turned out to be incomplete or have meaningless jargon.

In [96]:
# filter out articles having less than 10 characters
filtered_article = [a for a in newsfeed_article if len(a.split()) > 10]
len(filtered_article)

24404

Now that we have the approved list of news articles, we can then tokenize them. This process will further filter out noisy or spurious tokens.

#### Tokenization and Vectorization

As part of the tokenization process, I also included domain-specific stop-words that better filter out irrelevant words from the corpus. The tokenizer also handles unicode characters, space, punctuation and stop words.

In [45]:
# load spacy and disable unnecessary components
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

# add custom stop words
custom_stop = {'say', 'publication', 'publish', 'time', 'date', 'news', 'story', 'newspaper', 'vanguard', 'copyright', 'according', 'year',
               'month', 'day', 'week', 'particular', 'particularly', 'statement', 'report', 'reporter', 'doubt', 'appear', 
               'likely', 'unlikely', 'probably', 'occur', 'cause', 'find',}

nlp.Defaults.stop_words.union(custom_stop)

# building the tokenizer function
def custom_tokenizer(text):
    doc = nlp(text)
    return [
        t.lemma_.lower() for t in doc
                        if t.is_alpha 
                        and t.is_ascii
                        and not t.is_stop
                        and not t.is_punct 
                        and not t.is_space
                        and t.lemma_.lower() not in custom_stop]

Now that the function has been defined and metadata stripped, I applied the tokenizer.

In [47]:
%%time
tokenized_articles = list(map(custom_tokenizer, nlp.pipe(filtered_article, n_process=4)))
print(tokenized_articles[0][:100])

['armed', 'conflict', 'information', 'reveal', 'attack', 'security', 'force', 'location', 'vicinity', 'village', 'engage', 'security', 'personnel', 'exchange', 'gunfire', 'attacker', 'repel', 'security', 'force', 'reinforcement', 'town', 'arrive', 'area', 'casualty', 'unknown', 'unknown', 'indicate', 'security', 'force', 'conduct', 'clearance', 'operation', 'village', 'troop', 'neutralise', 'unknown', 'number', 'bh', 'iswap', 'member', 'rescue', 'unknown', 'number', 'civilian', 'trap', 'village', 'casualty', 'figure', 'operation', 'receive', 'source', 'indicate', 'conduct', 'incursion', 'village', 'engage', 'security', 'personnel', 'exchange', 'gunfire', 'attack', 'repel', 'theattacker', 'withdraw', 'unknown', 'location', 'number', 'casualty', 'unknown', 'accord', 'information', 'receive', 'attack', 'security', 'force', 'deployment', 'area', 'village', 'engage', 'security', 'personnel', 'exchange', 'gunfire', 'attack', 'repel', 'securityforce', 'attend', 'scene', 'engage', 'attacker', 

In [48]:
# create a Gensim Dictionary mapping words to unique IDs
dictionary = corpora.Dictionary(tokenized_articles)
len(dictionary)

13837

Before training, we attempt to further prune uncommon words using the _filter_extremes_ method. This specifically flags and trims low and high-frequency words from the corpus. This would help to optimize storage and processing speed.

In [49]:
dictionary.filter_extremes(no_below=5, no_above=0.5)
len(dictionary)

5903

The dictionary size significantly reduced further optimizing the performance.

To implement the model, I computed a bag-of-words table. This essentially is an array that shows the frequency of tokens for every article in the corpus.

In [50]:
# convert tokenized texts into standard Bag-of-Words (Frequency Counts)
bow_corpus = [dictionary.doc2bow(article) for article in tokenized_articles]

The entire corpus has 10,098 words distributed across 24,404 articles. Now to model building.

#### Topic Modelling

Now that all pre-processing is done, we can implement the topic model. Considering the dictionary size, I chose to segment the corpus into 11 topics. Choosing the ideal number of articles is a more of a science. I had to iterate severally to land on 11 topics. This was chosen because it applies a clear delineation across the topics with minimal noise.
<br>In addition, the model's hyperparameters _alpha_ and _eta_ parameters are both set to **auto**. The former controls how topics are distributed across the documents while the latter defines word distribution per each topic.

In [66]:
%%time

# configuring the topic model
NUM_TOPICS = 11
lda_model = models.LdaModel(corpus=bow_corpus, num_topics=NUM_TOPICS, id2word=dictionary,\
                alpha='auto', eta='auto', passes=15, random_state=1)

CPU times: total: 53.9 s
Wall time: 4min 1s


In [67]:
# show alpha and eta for the model
print(lda_model.alpha)
print(lda_model.eta)

[0.8324693  0.45744097 0.2704444  0.712968   1.0170348  0.26444998
 0.25908127 0.28195858 0.20740342 0.5896819  0.2910676 ]
[0.0728044  0.10206536 0.07337326 ... 0.06884533 0.07171097 0.06948116]


From the above, _alpha_ and _beta_ values are relatively low indicating a small distribution of topics per document and words per topic respectively.

In [69]:
lda_model.print_topics()

[(0,
  '0.032*"police" + 0.029*"state" + 0.026*"kill" + 0.025*"area" + 0.024*"attack" + 0.022*"suspect" + 0.019*"member" + 0.017*"people" + 0.017*"arrest" + 0.016*"community"'),
 (1,
  '0.026*"oil" + 0.013*"need" + 0.011*"technology" + 0.011*"sector" + 0.010*"country" + 0.010*"service" + 0.010*"new" + 0.010*"likelihood" + 0.010*"development" + 0.010*"ruling"'),
 (2,
  '0.037*"anticipate" + 0.027*"people" + 0.026*"city" + 0.023*"protester" + 0.021*"worker" + 0.019*"airport" + 0.016*"capital" + 0.015*"child" + 0.013*"aid" + 0.012*"prompt"'),
 (3,
  '0.056*"attack" + 0.030*"military" + 0.025*"group" + 0.025*"force" + 0.025*"kill" + 0.024*"terrorist" + 0.018*"soldier" + 0.017*"operation" + 0.016*"militant" + 0.016*"target"'),
 (4,
  '0.025*"government" + 0.021*"protest" + 0.020*"state" + 0.019*"election" + 0.017*"country" + 0.015*"opposition" + 0.013*"security" + 0.013*"political" + 0.013*"violence" + 0.011*"group"'),
 (5,
  '0.023*"disclose" + 0.015*"strategy" + 0.014*"give" + 0.012*"rela

Some of the topics appear to be related with some distinction. There is some noise present. This is often unavoidable but the number of topics K was chosen to minimize such.   
To improve topic interpretability, I prepared a topic label as that is outside the scope of LDA capabilities.

In [70]:
# semantically labelling the extracted topics
topic_map = {
    0: "Crime & Security",
    1: "Energy & Oil Sector",
    2: "Civil Unrest",
    3: "Insurgency & Military Ops",
    4: "Election Unrest & Violence",
    5: "Geopolitics & International Events",
    6: "Structural Collapse",
    7: "Climate Change & Public Health",
    8: "Pipeline Vandalism and Accident",
    9: "Human Rights & Law",
    10: "Road Accident & Casualities"
}

### Testing the Topic Model

Now I can use the model to classify new articles and assign them a topic. I built a function to accomplish this.  
It follows the same established workflow of tokenizing and BOW generation to identifying the predominant topics and outputting the results in a readable format.

In [78]:
def lda_classifier(article, min_prob=0.15, lda_model=lda_model):
    # tokenize article
    new_article = custom_tokenizer(article)

    # implement bow on article
    new_bow = dictionary.doc2bow(new_article)

    # get document topics (sorted descending)
    topic_prob_pairs = sorted(
        lda_model.get_document_topics(new_bow, minimum_probability=0.0),
        key=lambda tup: tup[1],
        reverse=True,
    )

    # guard clause for empty tokens/BOW
    if not topic_prob_pairs:
        return pd.DataFrame([{
            'Primary Topic ID': None,
            'Primary Topic Label': None,
            'Primary Probability': None,
            'Secondary Topic ID': None,
            'Secondary Topic Label': None,
            'Secondary Probability': None,
        }])

    # extract primary topic and probability
    prim_id, prim_prob = topic_prob_pairs[0]

    # extract secondary topic
    sec_id, sec_label, sec_prob = None, None, None
    if len(topic_prob_pairs) > 1:
        candidate_id, candidate_prob = topic_prob_pairs[1]
        if candidate_prob >= min_prob:
            sec_id = candidate_id
            sec_prob = round(float(candidate_prob), 4)
            sec_label = topic_map.get(sec_id, 'Unmapped Topic')

    # define the output structure
    data = {
        'Primary Topic ID': prim_id,
        'Primary Topic Label': topic_map.get(prim_id, 'Unmapped Topic'),
        'Primary Probability': round(float(prim_prob), 4),
        'Secondary Topic Label': sec_label,
        'Secondary Probability': sec_prob,
    }

    return pd.DataFrame([data])

Now to test on some sample articles...

In [85]:
text = 'House of representatives member Farouk Lawan today arraigned before an FCT High court in Abuja for bribery and corruption. The house member face seven count charge Â for soliciting $3 million bribe from controversial buswinessman Femi Otedola.'

lda_classifier(text)

,Primary Topic ID,Primary Topic Label,Primary Probability,Secondary Topic Label,Secondary Probability
0,9,Human Rights & Law,0.2796,Election Unrest & Violence,0.2118


In [84]:
text = 'A Kenyan police officer in the Kapenguria police station in Western Kenya went on a killing rampage, shooting and killing six of his fellow police officers.The shooting began 5:20AM local time on Thursday, and led to an eight-hour siege that led to a standoff with elite policemen who had been flown in from Nairobi.The shooter, Abdilhakim Maslah, is believed to have committed the atrocity as an act of terrorism.A separate police statement to the media said the officer, for yet unknown reasons, went berserk and grabbed a firearm and started shooting. According to a statement from a policeman, Mr. Maslah wore a "turban that covered his whole face, leading to the previous suspicions that he was an extremist.The station commander was also one of those who was killed, according to West Pokot County Commissioner Wilson Wanyanga said.According to the BBC, a source said that the gunman was unhappy that his request to resign from the police had been denied as he had not served the requisite 10 years of service after his graduation in 2013'
lda_classifier(text)

,Primary Topic ID,Primary Topic Label,Primary Probability,Secondary Topic Label,Secondary Probability
0,0,Crime & Security,0.4969,None,None


In [86]:
text = 'With Rising Theft, Nigeria Records 193 million barrels of crude oil deficit in 11 monthsâ€. This is otherwise translated into an estimated $3.5 billion of revenue lost to crude oil theft in 2021 alone, in other words, about 10% of the countryâ€™s foreign reserves. For a country that depends on petroleum products for about 85% of its total exports revenue and has been unable to define a future for itself beyond oil, oil theft is akin to a national calamity, a massive erosion, and an economic sabotage of the highest order. Even if it may not be the only factor that contributes to crude oil deficit, its impact is worth investigating. With regard to oil theft, ThisDay newspaper was not exactly reporting any new trend. Oil theft has been perennial and unceasing and indeed, it gets worse by the year.In its latest audit report, made public in July 2021, the Nigeria Extractive Industries Transparency Initiative (NEITI) indicated that in 2019, Nigeria lost 42.25 million barrels of crude oil to oil theft, valued at 2.77 billion dollars.Â  This was actually meant to be an improvement (imagine!) because in 2018, 53.28 million barrels were stolen. And then in 2021, 193 million barrels of crude vanished from Nigeriaâ€™s resources. The value of stolen crude in Nigeria is enough to fix many of the countryâ€™s problems and reduce the obsession with borrowings. This is the reason why oil theft must be stopped.Â  On the average, Nigeria loses about 200, 000 barrels per day. What is stolen in concrete terms is not just crude oil, but jobs, opportunities, and possibilities. Oil theft is also a veritable example of grand corruption, and this is the point that has been made consistently in NEITIâ€™s audit reports. The opaqueness that dominates the entire oil and gas value chain in Nigeria accounts for oil theft and loss of revenue. The absence of political will to tackle the problem makes it worse. Shell, ExxonMobil, Chevron and Total divested from Nigeria in part because of oil theft.ImageIn September 2021, the Federal Government decided to set up an Inter-Ministerial Committee on the recovery of crude oil and illegally refined petroleum products in the Niger Delta Region comprising the Department of Petroleum Resources (DPR), the Nigeria National Petroleum Corporation (NNPC), the National Oil Spill Detection and Response Agency (NOSDRA), all backed by the security agencies â€“ the Nigerian Army, the Navy, the Nigeria Security and Civil Defence Corps (NSCDC) and others. The committeeâ€™s mandate is drawn from the provisions of the Assets Tracing, Recovery and Management Regulations 2019. Yet, by the end of the year, 193 million barrels of crude oil had disappeared and certainly that must be an under-valuation, an estimate.It is well known that there is no proper documentation of anything in Nigeria. We donâ€™t even know how many we are. The National Population Commission (NPC) has no accurate register of births and deaths. Should it therefore be any surprise that there is no mechanism in place for monitoring how many barrels of crude oil Nigeria produces or the exact volume of it that is sold? Three years ago, there was some fancy talk about the introduction of technology to monitor output and activities along Nigeriaâ€™s oil pipelines network to detect sabotage, human interference and protect critical infrastructure. Oil was discovered in Nigeria, in Oloibiri, Bayelsa state in 1956. In 2022, Nigeria is still talking about how to protect pipelines through the adoption of technology. Even if technology is deployed through automation, the internet of things, drone technology, and the electronic monitoring that certain commentators recommend would still be an excuse to award contracts and make more money. Whatever works in other countries, Nigeria takes the same ideas and turns them upside down.The people who want to stop oil theft are really not interested in stopping anything, so it seems, for indeed, oil theft is an organized crime, with a network of stakeholders that cuts across many layers of interest. And that includes the same agencies saddled with the responsibility to stop it. Illegal oil bunkering: hot tapping or cold tapping, or the smuggling and diversion of petroleum products is an expensive enterprise, that involves the collusion of both state officials and their agents.Â  It may not be incorrect to argue, in fact,Â  that nothing has been done because those who should take the decision or their agents are themselves involved, or they have been compromised. Crude oil in the international market has a signature imprint that indicates the source, but somehow, stolen crude from Nigeria simply disappears into a sinkhole, without trace. There are also illegal refineries in the Niger Delta. Every Minister of Mines and Steel Development develops a plan for addressing the menace of illegal refineries, but nothing ever gets done. Even Governors complain about illegal refineries.Most recently, on January 1, 2022, Governor Nyesom Wike of Rivers State, in his New Year address, devoted some paragraphs to the challenge of the environment in the state. He condemned the pollution of the environment by the operation of illegal refineries. He even knows their location: â€œillegal crude oil refining sites along Creek Road and adjoining areas of the cityâ€¦â€ and he wants them shut down with immediate effect. He added that all local government Chairmen should work with â€œcommunity leaders to locate and identify those behind illegal bunkering and crude oil refining sites in their localities and report to my office for further actionâ€¦â€ It would be a miracle indeed if either Wike or any other Governor in the Niger Delta would be able to put a stop to illegal oil bunkering activities in the entire region. The big-time oil bunkerers are major figures in the communities and key financiers of political processes!Oil theft is further tied to the politics of Nigeria and the ownership of mineral resources. Section 44(3) of the 1999 Constitution, item 39 Schedule II of the Exclusive Legislative List and Section 1 of the Petroleum Act, 1969 vests the ownership and control of natural resources in any part of Nigeria in the Federal Government for the benefit of the people. (Also see Attorney General of the Federation vs. AG Abia State). For decades, the people of the Niger Delta and others have argued that this is a departure from the Federal principle that Nigeria claims to embrace and that as operationalized, the Federal Governmentâ€™s ascribed ownership of mineral resources amounts to gross injustice more so as the Niger Delta which produces the mainstay of the economy remains dispossessed, marginalized and underdeveloped compared to other parts of the country that contribute less, and yet seek to control what does not belong to them.The battle over what is termed â€œResource Controlâ€ has taken many dimensions over the years including the agitations that led to the Willinks Commission Inquiry on Minority Rights of 1957/58, the heroism of Isaac Adaka Boro (1966), the Ogoni peopleâ€™s Struggle for Survival, the Kaiama Declaration, the politics of agitation for resource control, Niger Delta militancy and calls for a complete restructuring of Nigeria. In 2004, Niger Delta Representatives walked out of the National Political Reform Conference when a consensus could not be reached on the subject of resource control and derivation. Niger Delta activists have since taken up this matter by insisting that derivation is inadequate, development initiatives such as the Niger Delta Development Commission and OMPADEC before it, amount to mere tokenisms, and that total resource control is what the people want. This matter reared its head again recently, and apparently will never go away, given the North vs South alignment around it. Former President Olusegun Obasanjo had expressed the opinion at a public seminar that Nigeriaâ€™s crude oil does not belong to the people of the Niger Delta but to all Nigerians and to say anything to the contrary would be illegal and unconstitutional.President Obasanjo correctly stated the position of the Nigerian Constitution, but in so doing he stirred the hornetâ€™s nest as Niger Delta stakeholders led by Chief E.K. Clark attacked him as â€œan enemy of the Niger Deltaâ€. Niger Delta activists and many other Nigerians think that the 1999 Constitution is a dubious military decree that should not be quoted as Nigeriaâ€™s grund norm. President Obasanjoâ€™s argument must have reminded them of that other argument actively pushed by Northern intellectuals in the 80s and 90s that the oil in the Niger Delta actually came from Northern Nigeria and settled in the Delta, as part of a given process of geological sedimentation. The consensus in the Niger Delta is that this is a thiefâ€™s argument and that the real thieves of crude oil are those who exploit other peopleâ€™s resources and who then turn around to insult the real owners.It is also in this regard that local players in the Niger Delta who are involved in illegal oil bunkering do not consider their activities theft or crime. In a curious good thief vs. bad thief binary at the heart of oil politics in Nigeria, they justify their own oil theft, and openly flaunt their ill-gotten wealth because they believe that they cannot be taken to task for stealing what belongs to them, theirÂ  grandfathers and generations yet unborn. They find ready allies across Nigeria and the rest of the world, because everyone else is anxious to make a quick buck. Many young persons in the Niger Delta would rather be a militant or an oil thief. Thievery, by the way, is a national pastime, a national creed, in Nigeria. Everybody is looking for something to steal: from gold in Zamfara and Ilesa to bitumen in Ondo, crude oil in the Niger Delta Basin, and the vaults of the Central Bank, if possible.The economics, mathematics, cost and politics of oil theft point in one direction: the need for the country to put in place a strong surveillance mechanism. The countryâ€™s pipeline network is decayed, hence making the work of the oil thief easy. There was a recent blow out at AITEOâ€™s OML 29 well-head in Santa Barbara River, Nembe, Bayelsa State. It took nearly a month for NOSDRA to be aware of it, and for any agency of government to take any action at all. It is important to have the necessary infrastructure and technology in place, and to treat oil theft strictly as economic sabotage. The penal structure for the crime should also be strengthened. Given the cost to the nation, the minimum sanction for oil theft should be life or death sentence. The politics of oil ownership or trusteeship cannot be advanced as an excuse for criminality. Until Nigeria decides to address its divisive politics, and public officials stop their hollow sloganeering about national unity, the resort to self-help, in form of oil theft or illegal bunkering, should not replace the rule of law. Crude oil refining has also been a problem. When will Nigeriaâ€™s refineries begin to function again at optimal capacity and profitably too?II:Â Bolaji A. Akinyemi At 80It would require a whole Festschrift to capture the essence of Professor Bolaji Akinwande Akinyemi, who turns 80, today, Tuesday, January 4, 2022. Academic, author, public policy expert, distinguished Professor, man of letters, senior citizen, Professor Akinyemi is a Nigerian icon, one of the diamonds that continue to shine luminously in the Nigerian landscape and whose engagements with his country and the international community, through his writings, and policy interventions confirm his genius, humanism and excellence. In 1975, he was a 33-year old Political Science lecturer at the University of Ibadan when he was appointed as the Director-General of the Nigeria Institute of International Affairs (NIIA,) Nigeriaâ€™s foreign policy think tank. Akinyemi brought not just youthful energy to the NIIA, he imbued the Institute with his exceptional brain power and intellection and left behind a legacy, upon which his successors, added their own blocks.His reward was his retention in that position, over a period of eight years, by a total of three administrations: Murtala Muhammad, Olusegun Obasanjo and Shehu Shagari and his subsequent appointment as Nigeriaâ€™s Minister of Foreign Affairs by General Ibrahim Babangida. Akinyemi was a creative thinker coming up, at every turn with original ideas: the Concert of Medium Powers,Â  the Black Bomb, Nigerian exceptionalism, the power school, and although many of his ideas were rigorously debated, he was never found wanting whenever he was dragged into the arena of intellectual pugilism. I recall his exchanges with the polemicist and poet, the inimitable Odia Ofeimun. Akinyemi argued for the authenticity of African identity and ideas, and the continent as a frontier for development. It was during his time as Nigeriaâ€™s Minister of Foreign Affairs that the Technical Aid Corps, (the TAC)) was conceived and established.He has since through all seasons remained active in the public domain, an affirmation of his top credentials as a public intellectual, and as a symbol of how intellectuals can with the power of ideas forge a necessary link between the world of ideas and the world of action, between theory and praxis. In this regard, Akinyemi is a master of the art of tact, balance and ambidextrous navigation, as he cultivates the persona of an insider who is yet an outsider, a key player within the establishment at various times, and yet an activist for public good in defence of the masses. The former DG NIIA, and former Minister, during Nigeriaâ€™s turbulent years of transition from military rule to civilian rule, joined Nigeriaâ€™s Democratic Coalition and lent his voice openly to the struggle to save Nigeria. He remains fully engaged in public affairs, both locally and internationally, refusing to slow down. His capacity for work-life balance is also impressive.In those days, he used to show up now and then at the Niteshift Coliseum, a night-club and entertainment centre, a stone-throw away from his Opebi residence, where he mingled with the young, and enjoyed the good life. He later joined us at The Guardian, on special invitation, as a Consultant to The Guardian Editorial Board. It was a pleasure working with him. These days, Professor Akinyemi has also been a major go-to person for us at Arise News TV for commentaries on international affairs. He never disappoints. Close to 50 years in the public arena, Professor Akinyemi has remained prodigious and intellectually formidable. I am tempted to say that they donâ€™t make them like that anymore, but Professor Akinyemi himself would be the first to correct me on that score, because indeed what his generation has done is to inspire, amidst the rot that has enveloped Nigeria, a younger generation in academia and civil society, who continue to raise hopes about Nigeriaâ€™s future.I have no doubts that Professor Bolaji Akinyemi is fully aware that there are implications to his attainment of the age of 80. I look forward to seeing him soon in his signature bow-tie and bespoke suit, to toast to his life and times and distinction. Happy Birthday, sir. Many Happy Returns.'
lda_classifier(text)

,Primary Topic ID,Primary Topic Label,Primary Probability,Secondary Topic Label,Secondary Probability
0,1,Energy & Oil Sector,0.283,Election Unrest & Violence,0.185


The results look fairly solid. The output demonstrates high precision for primary topics and elements of the secondary topic (especially Violence) was captured.  
The model can effectively double as a topic classifier.

### Model Visualization

To visualize the results, I employed the famed pyLDAvis library. This tool was built specifically for topic modelling. It comes with an advanced interactive capability with the option of tweaking displayed information.

In [93]:
from IPython.display import HTML, display
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# 1. Prepare data
vis_data = gensimvis.prepare(lda_model, bow_corpus, dictionary, sort_topics=False)

# 2. Render as raw HTML frame (bypasses RequireJS)
html_string = pyLDAvis.prepared_data_to_html(vis_data)
display(HTML(html_string))

In [95]:
# Run this to generate a standalone HTML file
html_string = pyLDAvis.prepared_data_to_html(vis_data)

with open('lda_topic_visualization.html', 'w', encoding='utf-8') as f:
    f.write(html_string)

We can observe that topics are fairly distributed with varying sizes (the size of a topic bubble represents the prevalence of topics in the overall corpus). Hovering or selecting the topic bubbles highlight the dominant terms. The same applies to the term frequency chart i.e it highlights the topics where the term applies.
<br>Topics 5("Election Unrest & Violence"), 4("Insurgency & Military Ops") and 1("Crime & Security") are the most dominant. 
<br>On the other hand 6("Geopolitics & International Events") and 7("Structural Collapse") are the least represented.

### Conclusion

In essence, Topic modelling is a fascinating tool for capturing the dominant themes that are present in documents. It is a great practice for analyzing and parsing large documents. 
This was an excelent learning opportunity. I look forward to deploying this tool to new applications.